# 1회차 실습: Vector(벡터)·Linear combination(선형결합) ($\mathbb{R}^n$)

> Part 1: 1회차 (Vector·$\mathbb{R}^n$·덧셈·Scalar곱·Linear combination·Span 직관)
> 사전 reading: Strang §1.1 / 3Blue1Brown EoLA Ch.1

이 노트북은 강의교안 1회차의 흐름(B 섹션 Vector·두 연산 → C 섹션 Linear combination·Span → D 섹션 AI 적용)을 그대로 따라갑니다. 매 절은 **Definition → (Theorem) → Application** 순서로 정리합니다.

## 학습 목표

이번 실습이 끝나면 다음을 NumPy 코드로 직접 보일 수 있습니다.

1. $\mathbb{R}^n$의 Vector 객체를 생성하고 차원(shape)을 확인합니다.
2. **덧셈**($\mathbf{u}+\mathbf{v}$)·**Scalar곱**($\alpha\mathbf{x}$) 두 연산을 성분별로 수행합니다.
3. **Linear combination** $\sum \alpha_i \mathbf{v}_i$를 직접 계산하고, 표준 basis $\mathbf{e}_1, \ldots, \mathbf{e}_n$ 분해를 확인합니다.
4. **Span**의 직관(직선·평면·전체)을 $\mathbb{R}^3$ 예제로 봅니다.
5. **신경망 한 층 = 가중 Linear combination**임을 4-차원 미니 선형층으로 확인합니다 (MNIST 응용은 2회차).

### 정의될 객체 목록

| 번호 | 객체 |
|---|---|
| 정의 1.1 | $\mathbb{R}^n$의 Vector |
| 정의 1.2 | Vector 덧셈 |
| 정의 1.3 | Scalar(스칼라)곱 |
| 정의 1.4 | Linear combination |
| 정의 1.5 | Span (직관) |

### 사용 라이브러리

- NumPy, Matplotlib

In [ ]:
# Colab 한글 폰트 설정 (matplotlib 깨짐 방지)
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # 한글 폰트 자동 등록 (NanumGothic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print('NumPy version:', np.__version__)

## 1. Definition: $\mathbb{R}^n$의 Vector (정의 1.1)

$n$개 실수 순서쌍을 한 객체로 묶은 것이 $\mathbb{R}^n$의 Vector입니다. 이 강의의 기본 표기는 **Column vector(열벡터)**.

$$\mathbf{x} = \begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix} \in \mathbb{R}^n, \qquad \mathbf{x}^\top = (x_1, x_2, \ldots, x_n)$$

- $x_i \in \mathbb{R}$를 **성분(component)** 또는 **좌표(coordinate)**라 부릅니다.
- $n$을 **차원(dimension)**이라 부릅니다.
- 표기 관례: Vector는 **굵게**($\mathbf{x}$), Scalar는 보통체($\alpha, x_i$).

In [ ]:
# R^4의 두 Vector
a = np.array([3, 1, -2, 4], dtype=float)
b = np.array([-1, 2, 5, 0], dtype=float)

print('a       =', a)
print('b       =', b)
print('a.shape =', a.shape, ' (R^4 → (4,))')
print('a.ndim  =', a.ndim, '   (1차원 배열 = Column·Row 미구분 형태)')
print('차원 n  =', a.size)

## 2. Definition: 덧셈·Scalar곱 (정의 1.2·1.3)

### 정의 1.2 (덧셈)
같은 차원의 두 Vector의 합은 **성분별 합**입니다.
$$\mathbf{x} + \mathbf{y} = (x_1+y_1,\ x_2+y_2,\ \ldots,\ x_n+y_n)^\top$$
차원이 다르면 합이 정의되지 않습니다.

### 정의 1.3 (Scalar곱)
실수 $\alpha \in \mathbb{R}$와 Vector $\mathbf{x} \in \mathbb{R}^n$의 곱은 **성분별 Scalar 곱**입니다.
$$\alpha\mathbf{x} = (\alpha x_1,\ \alpha x_2,\ \ldots,\ \alpha x_n)^\top$$

In [ ]:
# 덧셈·Scalar곱·두 연산을 섞은 식 (Linear combination 예고)
print('a + b      =', a + b)
print('2 * a      =', 2 * a)
print('2a - 3b    =', 2 * a - 3 * b)  # 두 연산을 섞은 식

# 자연스러운 성질 수치 검증 (강의교안 B-5: 7회차에서 axioms로 격상)
u = np.array([1.0, 2.0, 3.0])
v = np.array([-1.0, 0.5, 2.0])
w = np.array([4.0, -2.0, 1.0])

assert np.allclose(u + v, v + u),                       '교환법칙'
assert np.allclose((u + v) + w, u + (v + w)),           '결합법칙'
assert np.allclose(u + np.zeros_like(u), u),            '덧셈 항등원'
assert np.allclose(u + (-u), np.zeros_like(u)),         '역원'
assert np.allclose(2 * (3 * u), (2 * 3) * u),           'Scalar 결합'
assert np.allclose(2 * (u + v), 2 * u + 2 * v),         '분배법칙'
print('✓ 덧셈·Scalar곱의 자연 성질 6개 모두 성립 (R^n에서 검증)')

## 3. Definition: Linear combination·Span (정의 1.4·1.5)

### 정의 1.4 (Linear combination)
Vector $\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k \in \mathbb{R}^n$, Scalar $\alpha_1, \ldots, \alpha_k \in \mathbb{R}$에 대해
$$\alpha_1 \mathbf{v}_1 + \alpha_2 \mathbf{v}_2 + \cdots + \alpha_k \mathbf{v}_k \;\in\; \mathbb{R}^n$$
를 **Linear combination(선형결합)**이라 부릅니다. $\alpha_i$를 **계수(coefficient)**라 합니다.

### 정의 1.5 (Span, 직관)
$$\mathrm{span}(\mathbf{v}_1, \ldots, \mathbf{v}_k) = \left\{ \alpha_1\mathbf{v}_1 + \cdots + \alpha_k\mathbf{v}_k \;:\; \alpha_i \in \mathbb{R} \right\}$$
주어진 Vector들로 **만들 수 있는 모든 결과의 집합**입니다 (정식 정의는 8회차).

### 관찰: 표준 basis 분해
$\mathbb{R}^3$의 임의 Vector $\mathbf{x} = (x_1, x_2, x_3)^\top$는
$$\mathbf{x} = x_1\mathbf{e}_1 + x_2\mathbf{e}_2 + x_3\mathbf{e}_3$$
로 표현됩니다 (좌표 자체가 표준 basis의 계수).

In [ ]:
# 표준 basis e_1, e_2, e_3: np.eye의 각 열이 e_i
I3 = np.eye(3)
e1, e2, e3 = I3[:, 0], I3[:, 1], I3[:, 2]
print('e_1 =', e1, ',  e_2 =', e2, ',  e_3 =', e3)

# 임의 좌표 (2, -3, 5) → 표준 basis Linear combination
x = 2 * e1 + (-3) * e2 + 5 * e3
print('\n2·e_1 - 3·e_2 + 5·e_3 =', x)
assert np.allclose(x, np.array([2.0, -3.0, 5.0])), '좌표 = 표준 basis 계수'
print('✓ 좌표 (2, -3, 5)^T 자체가 표준 basis의 Linear combination 계수')

# Span 직관: e_1, e_2만으로 만들 수 있는 모든 결과는 xy 평면
print('\nspan(e_1, e_2) 위의 점 예시 (z 좌표가 항상 0):')
for alpha, beta in [(1, 0), (0.5, -2), (-3, 4)]:
    p = alpha * e1 + beta * e2
    print(f'  {alpha}·e_1 + {beta}·e_2 = {p}')
print('→ 어떤 계수를 잡아도 z=0 → span(e_1, e_2) = xy 평면 (R^3의 한 평면)')

### 3.1 Span 모양 식별: 그림과 함께

강의교안 C-4의 예제를 그림으로 확인합니다.

- $\mathrm{span}((1, 1)^\top, (2, 2)^\top)$ = $(1,1)^\top$ 방향 **직선** (두 Vector가 평행)
- $\mathrm{span}((1, 0)^\top, (0, 1)^\top)$ = **$\mathbb{R}^2$ 전체** (모든 점)

**Vector 개수 ≠ Span 차원**입니다, 진짜 다른 방향의 개수가 차원입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
alphas = np.linspace(-2, 2, 80)

# 좌: 평행한 두 Vector: Span은 한 직선
v1 = np.array([1.0, 1.0])
v2 = np.array([2.0, 2.0])
pts = np.array([a1 * v1 + a2 * v2 for a1 in alphas for a2 in alphas])
axes[0].scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
axes[0].quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='red')
axes[0].quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='blue')
axes[0].set_title('span((1,1), (2,2)) = 직선 (평행)')
axes[0].set_xlim(-6, 6); axes[0].set_ylim(-6, 6); axes[0].set_aspect('equal'); axes[0].grid(alpha=0.3)

# 우: 독립인 두 Vector: Span은 R^2 전체
v1 = np.array([1.0, 0.0])
v2 = np.array([0.0, 1.0])
pts = np.array([a1 * v1 + a2 * v2 for a1 in alphas for a2 in alphas])
axes[1].scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
axes[1].quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='red')
axes[1].quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='blue')
axes[1].set_title('span(e_1, e_2) = R^2 전체')
axes[1].set_xlim(-2.5, 2.5); axes[1].set_ylim(-2.5, 2.5); axes[1].set_aspect('equal'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('관찰: 두 Vector가 평행하면 Span은 직선. 진짜 다른 방향이 둘일 때 Span이 평면.')

## 4. Application: 신경망 한 층 = 가중 Linear combination

강의교안 D-5에서 본 사실: 가장 간단한 인공 뉴런 한 개의 출력은
$$y = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b$$
출력이 여러 개인 한 층은
$$\mathbf{y} = W\mathbf{x} + \mathbf{b}$$
여기서 **$W\mathbf{x}$는 $W$의 열들의 Linear combination** (3회차 Column picture).

여기서는 **$W \in \mathbb{R}^{4 \times 3}$의 미니 선형층**으로 두 해석이 같음을 확인합니다.

- 입력 $\mathbf{x} \in \mathbb{R}^3$ (정의 1.1)
- 가중치 행렬의 **각 열** = 한 입력 성분이 출력 4개에 끼치는 영향 패턴 (정의 1.4: Linear combination)
- 출력 $\mathbf{y} \in \mathbb{R}^4$ = 세 열의 Linear combination (계수가 $x_1, x_2, x_3$)

> MNIST 같은 실제 데이터 응용은 2회차 노트북에서 다룹니다. 이번 회차는 두 연산·Linear combination이 신경망 한 줄에 어떻게 등장하는지만 봅니다.

In [ ]:
# 미니 선형층: 입력 3차원 → 출력 4차원
W = np.array([
    [ 1.0,  0.5, -1.0],
    [-2.0,  1.0,  0.5],
    [ 0.5,  2.0,  1.0],
    [ 1.0, -1.0,  0.5],
])  # shape (4, 3): 행: 출력 뉴런 4개, 열: 입력 성분 3개
b = np.array([0.1, 0.0, -0.2, 0.3])  # bias
x = np.array([2.0, -1.0, 3.0])       # 입력 vector

# 방법 A: 행렬·벡터 곱 (한 줄)
y_matrix = W @ x + b

# 방법 B: 세 열의 Linear combination 직접
col1, col2, col3 = W[:, 0], W[:, 1], W[:, 2]
y_lincomb = x[0] * col1 + x[1] * col2 + x[2] * col3 + b

print('y (행렬·벡터 곱)        =', y_matrix)
print('y (열의 Linear combination) =', y_lincomb)
assert np.allclose(y_matrix, y_lincomb), '두 해석은 같은 결과'
print('✓ Wx + b = (W의 열들의 x계수 Linear combination) + b')
print('  → 신경망 한 층의 본질이 곧 오늘 본 정의 1.4 Linear combination입니다.')

## 5. 연습 (자가 점검)

다음을 코드 셀에 직접 시도해 보고 결과를 본인 노트에 기록합니다.

### 연습 1 (덧셈의 교환·결합 법칙)
임의의 $\mathbf{u}, \mathbf{v}, \mathbf{w} \in \mathbb{R}^5$를 `np.random.randn(5)`로 생성하여 다음을 `np.allclose`로 확인하세요.

- $\mathbf{u} + \mathbf{v} = \mathbf{v} + \mathbf{u}$
- $(\mathbf{u} + \mathbf{v}) + \mathbf{w} = \mathbf{u} + (\mathbf{v} + \mathbf{w})$

### 연습 2 (Scalar곱의 분배법칙)
임의의 $\alpha, \beta \in \mathbb{R}$, $\mathbf{u}, \mathbf{v} \in \mathbb{R}^4$로 다음을 검증하세요.

- $(\alpha + \beta)\mathbf{u} = \alpha\mathbf{u} + \beta\mathbf{u}$
- $\alpha(\mathbf{u} + \mathbf{v}) = \alpha\mathbf{u} + \alpha\mathbf{v}$

### 연습 3 (표준 basis 분해)
$\mathbb{R}^5$에서 임의의 Vector $\mathbf{x}$를 만들고, 그 좌표 $x_i$를 사용해 $\mathbf{x} = \sum_{i=1}^5 x_i \mathbf{e}_i$로 재구성한 결과가 원본과 같은지 확인하세요.

### 연습 4 (Span 모양 식별)
다음 Span을 그림 또는 말로 묘사하세요 (직선·평면·공간 전체·한 점 중 어느 것?).

- (i) $\mathrm{span}((1, 0, 0)^\top, (0, 1, 0)^\top, (1, 1, 0)^\top) \subset \mathbb{R}^3$
- (ii) $\mathrm{span}((1, 2, 3)^\top) \subset \mathbb{R}^3$
- (iii) $\mathrm{span}(\mathbf{0}) \subset \mathbb{R}^n$

> 정답·풀이는 과제 1회차 제출본에 포함합니다.

## 6. 정리

오늘 도입한 정의 목록:

| 정의 | 내용 |
|---|---|
| 정의 1.1 | $\mathbb{R}^n$의 Vector: $n$개 실수 순서쌍 |
| 정의 1.2 | 덧셈: 성분별 합 |
| 정의 1.3 | Scalar곱: 성분별 Scalar 곱 |
| 정의 1.4 | Linear combination: $\sum \alpha_i \mathbf{v}_i$ |
| 정의 1.5 | Span (직관): 모든 Linear combination의 집합 |

오늘 검증한 사실:

- $\mathbb{R}^n$에서 덧셈·Scalar곱은 **자연스러운 6가지 성질**을 만족합니다 (교환·결합·덧셈 항등원·역원·Scalar 결합·분배). 6회차에서 일반 vector space의 8 axioms로 격상됩니다.
- 임의의 $\mathbf{x} \in \mathbb{R}^n$가 **표준 basis $\mathbf{e}_1, \ldots, \mathbf{e}_n$의 Linear combination**으로 표현됩니다. 좌표가 곧 계수.
- 두 Vector가 평행이면 Span은 직선, 진짜 다른 방향이 둘이면 평면. **Vector 개수 ≠ Span 차원**.
- **신경망 한 층 = $W$의 열들의 가중 Linear combination + bias**: 정의 1.4가 그대로 AI의 한 줄.

### 다음 회차(2회차)로 가는 다리

오늘은 한 Vector·여러 Vector를 다루는 두 연산을 봤습니다, 그러나 아직 **두 Vector가 얼마나 비슷한가**를 잴 수 없습니다. 2회차에서 **Norm(길이)**과 **Dot product(내적)**라는 두 도구를 도입하여 거리·각도·코사인 유사도를 정의합니다. MNIST 이미지 비교가 그때 본격 등장합니다.